In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Load Project Utilities & Initialize Notebook Widgets

In [0]:
%run /Workspace/consolidated_pipeline/consolidate_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Souce')


catalog=dbutils.widgets.get('catalog')
data_source= dbutils.widgets.get('data_source')

base_path=f's3://sportsbar-pro/{data_source}'
landing_path=f"{base_path}/landing/"
processed_path=f"{base_path}/processsed/"

print('Base Path: ',base_path)
print('Landing Path: ',landing_path)
print('Processed Path: ',processed_path)


bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.db_fact_{data_source}"

print('bronze table: ',bronze_table)
print('silver table: ',silver_table)
print('gold table: ',gold_table)

In [0]:
df=spark.read.option('header',True).option('inferSchema',True).csv(f'{landing_path}*.csv').withColumn('read_timestamp',F.current_timestamp()).select("*",'_metadata.file_name','_metadata.file_size')

print('Total Rows: ',df.count())
df.show()

In [0]:
(
    df.write.format('delta')
    .option('delta.enableChangeDataFeed','true')
    .mode('append')
    .saveAsTable(bronze_table)
)

In [0]:
files=dbutils.fs.ls(landing_path)
# files

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

### Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

### Transfromations

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
df_orders.show(100)

In [0]:
# check what's the maximun and minimum date

df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()